# 降维（Dimensionality Reduction）

对应课程：`phases/01-math-foundations/10-dimensionality-reduction`

> 高维数据自有其结构。选对观察的角度，你就能发现它。

本 notebook 把 `dim_reduction.py` 里的核心函数拆开：每个函数一组中文注释，后面跟一小段可运行实验。完整打印型 demo 仍在 `dim_reduction.py`。

**贯穿全课的模式：** 中心化 → 协方差 / 核矩阵 → 特征值排序 → 丢掉小方向。本课只用 numpy，不做 MNIST / sklearn / t-SNE / UMAP 下载。


## 学习目标（Learning Objectives）

- 从零实现 PCA：中心化、协方差、特征分解、投影
- 用解释方差比和肘部法则选主成分个数
- 比较 PCA / t-SNE / UMAP 做 2D 可视化的取舍（本 notebook 不下载 MNIST）
- 用 RBF 核 PCA 分开线性 PCA 分不开的非线性结构


## 0. 依赖

numpy 在课程允许清单里。不导入 sklearn，不拉取远程数据。


In [1]:
import numpy as np


## 1. PCA：找到方差最大的正交方向

算法：

1. 每列减均值（中心化）
2. 协方差矩阵 $\mathrm{Cov} = \frac{1}{n-1}X^\top X$（`np.cov` 默认如此）
3. 对称特征值分解；按特征值从大到小排
4. 取前 $k$ 个特征向量当投影轴
5. 解释方差比 = 该特征值 / 全部特征值之和

`components` 存成 $(k, d)$，与 sklearn 一致：`transform` 是 $(X-\mu)W^\top$。


In [2]:
class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None            # (k, d) 主成分，每行一个方向
        self.mean = None
        self.eigenvalues = None           # 前 k 个特征值（方差）
        self.explained_variance_ratio_ = None

    def fit(self, X):
        """中心化 → 协方差 → eigh → 按特征值降序取前 k。"""
        self.mean = np.mean(X, axis=0)
        X_centered = X - self.mean

        cov_matrix = np.cov(X_centered, rowvar=False)

        # 协方差对称，eigh 比 eig 更稳；返回升序，后面再翻转
        eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

        sorted_idx = np.argsort(eigenvalues)[::-1]
        eigenvalues = eigenvalues[sorted_idx]
        eigenvectors = eigenvectors[:, sorted_idx]

        self.components = eigenvectors[:, : self.n_components].T
        self.eigenvalues = eigenvalues[: self.n_components]
        total_var = np.sum(eigenvalues)
        self.explained_variance_ratio_ = self.eigenvalues / total_var

        return self

    def transform(self, X):
        """投影到前 k 个主成分： (X-mean) @ W.T"""
        X_centered = X - self.mean
        return X_centered @ self.components.T

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

    def inverse_transform(self, X_reduced):
        """从 k 维抬回原空间： X_hat = Z @ W + mean"""
        return X_reduced @ self.components + self.mean


## 2. 实验：一根细轴的 3D 数据

$x_1,x_2$ 走圆，$x_3 \approx 0.5 x_1 + 0.3 x_2$。第三维几乎是前两维的线性组合，真正的自由度大约是 2。PCA 应把绝大部分方差放进前两个成分。


In [3]:
np.random.seed(42)
n_samples = 500
t = np.random.uniform(0, 2 * np.pi, n_samples)
x1 = 3 * np.cos(t) + np.random.normal(0, 0.2, n_samples)
x2 = 3 * np.sin(t) + np.random.normal(0, 0.2, n_samples)
# 细维度：几乎落在前两维张成的平面上
x3 = 0.5 * x1 + 0.3 * x2 + np.random.normal(0, 0.1, n_samples)
X3d = np.column_stack([x1, x2, x3])

pca2 = PCA(n_components=2)
Z = pca2.fit_transform(X3d)
X_hat = pca2.inverse_transform(Z)

print("原始形状:", X3d.shape, "  降维后:", Z.shape)
print("explained_variance_ratio_:", pca2.explained_variance_ratio_)
print("累计解释方差:", float(np.sum(pca2.explained_variance_ratio_)))
print("k=2 重构 MSE:", float(np.mean((X3d - X_hat) ** 2)))

pca_full = PCA(n_components=3)
pca_full.fit(X3d)
print("三个成分的解释方差比:", pca_full.explained_variance_ratio_)


原始形状: (500, 3)   降维后: (500, 2)
explained_variance_ratio_: [0.59004014 0.40926775]
累计解释方差: 0.9993078891928763
k=2 重构 MSE: 0.002423600184004863
三个成分的解释方差比: [0.59004014 0.40926775 0.00069211]


## 3. 重构误差

$$
\mathrm{MSE} = \frac{1}{Nd}\sum_{i,j}(X_{ij}-\hat X_{ij})^2
$$

PCA 丢掉的误差正好对应被丢弃的特征值之和。$k$ 取满时误差应接近 0。


In [4]:
def reconstruction_error(X, X_reconstructed):
    """逐元素 MSE。越小说明丢掉的方向越不重要。"""
    return np.mean((X - X_reconstructed) ** 2)


print("k=2 MSE =", float(reconstruction_error(X3d, X_hat)))
pca1 = PCA(n_components=1)
X1 = pca1.inverse_transform(pca1.fit_transform(X3d))
print("k=1 MSE =", float(reconstruction_error(X3d, X1)))
pca3 = PCA(n_components=3)
X3 = pca3.inverse_transform(pca3.fit_transform(X3d))
print("k=3 MSE =", float(reconstruction_error(X3d, X3)), "(应接近 0)")


k=2 MSE = 0.002423600184004863
k=1 MSE = 1.4355776370121967
k=3 MSE = 1.6279796023421646e-30 (应接近 0)


## 4. 实验：重构误差随 $k$ 下降

造 20 维数据，但只有 5 个潜在因子。过了 $k=5$，再加成分几乎只拟合噪声，MSE 趋于噪声水平。


In [5]:
np.random.seed(42)
n_samples, n_features, n_informative = 300, 20, 5
base = np.random.randn(n_samples, n_informative)
mixing = np.random.randn(n_informative, n_features)
noise = np.random.randn(n_samples, n_features) * 0.1
X = base @ mixing + noise

print(f"{'k':>4s}  {'recon MSE':>12s}  {'last ratio':>12s}  {'cumulative':>12s}")
for k in [1, 2, 3, 5, 10, 15, 20]:
    pca_k = PCA(n_components=k)
    reduced = pca_k.fit_transform(X)
    rec = pca_k.inverse_transform(reduced)
    mse = reconstruction_error(X, rec)
    cumulative = float(np.sum(pca_k.explained_variance_ratio_))
    last = float(pca_k.explained_variance_ratio_[-1])
    print(f"{k:>4d}  {mse:>12.6f}  {last:>12.6f}  {cumulative:>12.4f}")

print("潜在因子是 5 维：k=5 之后 MSE 接近噪声，再加成分收益很小。")


   k     recon MSE    last ratio    cumulative
   1      3.115103      0.428028        0.4280
   2      1.897699      0.223531        0.6516
   3      1.007323      0.163484        0.8150
   5      0.007469      0.056768        0.9986
  10      0.004396      0.000100        0.9992
  15      0.001907      0.000085        0.9996
  20      0.000000      0.000061        1.0000
潜在因子是 5 维：k=5 之后 MSE 接近噪声，再加成分收益很小。


## 5. 核 PCA：在核诱导的特征空间做 PCA

线性 PCA 只能旋转坐标系。同心圆环在任何直线上都叠在一起。核技巧不显式升维，只算 $K_{ij}=k(x_i,x_j)$：

$$
k_{\mathrm{RBF}}(x,y)=\exp(-\gamma\|x-y\|^2)
$$

然后对核矩阵中心化再特征值分解。结果没有直接的 `inverse_transform`（需要预像近似），所以这里只返回低维坐标。


In [6]:
def kernel_pca(X, n_components, kernel="rbf", gamma=1.0):
    """在中心化核矩阵上做 PCA，返回 n 个点的 k 维坐标。"""
    n = X.shape[0]

    if kernel == "rbf":
        # ||x-y||^2 = ||x||^2 + ||y||^2 - 2 x·y，避免双层循环
        sq_dists = (
            np.sum(X ** 2, axis=1).reshape(-1, 1)
            + np.sum(X ** 2, axis=1).reshape(1, -1)
            - 2 * X @ X.T
        )
        K = np.exp(-gamma * sq_dists)
    elif kernel == "poly":
        K = (X @ X.T + 1) ** gamma
    else:
        K = X @ X.T

    # 特征空间中心化：K <- (I-1/n) K (I-1/n)
    one_n = np.ones((n, n)) / n
    K_centered = K - one_n @ K - K @ one_n + one_n @ K @ one_n

    eigenvalues, eigenvectors = np.linalg.eigh(K_centered)
    sorted_idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[sorted_idx]
    eigenvectors = eigenvectors[:, sorted_idx]

    top_vals = eigenvalues[:n_components]
    top_vecs = eigenvectors[:, :n_components]

    # 特征向量按 1/sqrt(λ) 缩放，再乘回 λ 得到投影坐标
    for i in range(n_components):
        if top_vals[i] > 1e-10:
            top_vecs[:, i] = top_vecs[:, i] / np.sqrt(top_vals[i])

    return top_vecs * top_vals[:n_components]


## 6. 实验：两圈同心圆环（$N=200$）

内圈半径 1、外圈半径 3。线性 PCA 投影到 1 维后两圈区间重叠；RBF 核 PCA 把半径差映射成第一坐标上的分离。


In [7]:
np.random.seed(42)
n_per_ring = 100  # 两圈共 200 点，核矩阵 200x200
theta_inner = np.random.uniform(0, 2 * np.pi, n_per_ring)
r_inner = 1.0 + np.random.normal(0, 0.1, n_per_ring)
inner = np.column_stack([r_inner * np.cos(theta_inner), r_inner * np.sin(theta_inner)])

theta_outer = np.random.uniform(0, 2 * np.pi, n_per_ring)
r_outer = 3.0 + np.random.normal(0, 0.1, n_per_ring)
outer = np.column_stack([r_outer * np.cos(theta_outer), r_outer * np.sin(theta_outer)])

X_circles = np.vstack([inner, outer])
labels = np.array([0] * n_per_ring + [1] * n_per_ring)

pca_linear = PCA(n_components=1)
X_linear = pca_linear.fit_transform(X_circles)
inner_range = (float(X_linear[labels == 0].min()), float(X_linear[labels == 0].max()))
outer_range = (float(X_linear[labels == 1].min()), float(X_linear[labels == 1].max()))
overlap = inner_range[1] > outer_range[0] and outer_range[1] > inner_range[0]
print("线性 PCA 1 维")
print(f"  内圈范围: [{inner_range[0]:.2f}, {inner_range[1]:.2f}]")
print(f"  外圈范围: [{outer_range[0]:.2f}, {outer_range[1]:.2f}]")
print(f"  重叠: {overlap}（直线分不开圆环）")

X_kpca = kernel_pca(X_circles, n_components=2, kernel="rbf", gamma=0.5)
inner_mean = float(X_kpca[labels == 0, 0].mean())
outer_mean = float(X_kpca[labels == 1, 0].mean())
print("\n核 PCA (RBF, gamma=0.5)")
print(f"  内圈 PC1 均值: {inner_mean:.4f}")
print(f"  外圈 PC1 均值: {outer_mean:.4f}")
print(f"  PC1 分离: {abs(outer_mean - inner_mean):.4f}")

print("\ngamma 扫描:")
for g in [0.1, 0.5, 1.0, 5.0]:
    X_k = kernel_pca(X_circles, n_components=2, kernel="rbf", gamma=g)
    sep = abs(float(X_k[labels == 0, 0].mean() - X_k[labels == 1, 0].mean()))
    print(f"  gamma={g:<4}  PC1 分离={sep:.4f}")


线性 PCA 1 维
  内圈范围: [-1.11, 1.11]
  外圈范围: [-3.15, 3.06]
  重叠: True（直线分不开圆环）

核 PCA (RBF, gamma=0.5)
  内圈 PC1 均值: -0.3536
  外圈 PC1 均值: 0.3536
  PC1 分离: 0.7071

gamma 扫描:
  gamma=0.1   PC1 分离=0.1529
  gamma=0.5   PC1 分离=0.7071
  gamma=1.0   PC1 分离=0.2931
  gamma=5.0   PC1 分离=0.0910


## 6. PCA vs t-SNE vs UMAP（学习目标）

本课不拉 MNIST。三种方法的取舍：

| | 保住什么 | 失败模式 |
|--|----------|----------|
| PCA | 全局方差、可重建 | 非线性流形拧成一团 |
| t-SNE | 局部邻域，簇好看 | 轴无意义、不能轻易加新点、对 perplexity 敏感 |
| UMAP | 局部+更连贯的全局 | 仍是可视化工具，不是压缩编码 |

肘部：解释方差比不再明显下降的 $k$，就是该停的地方。上面的 recon MSE 曲线已经在用这件事。


In [8]:
print("PCA：线性、可逆、适合压缩和去相关")
print("t-SNE：保局部，适合看簇，不要拿它当特征")
print("UMAP：通常比 t-SNE 更快、全局更稳，同样不要当编码器")
print("选 k：看 explained_variance_ratio_ 的肘，而不是固定 2 维")


PCA：线性、可逆、适合压缩和去相关
t-SNE：保局部，适合看簇，不要拿它当特征
UMAP：通常比 t-SNE 更快、全局更稳，同样不要当编码器
选 k：看 explained_variance_ratio_ 的肘，而不是固定 2 维


## 对照表

| 函数 / 方法 | 角色 |
|-------------|------|
| `PCA.fit` | 中心化、协方差、`eigh`、按方差取前 $k$ |
| `PCA.transform` | 投影到主成分 |
| `PCA.inverse_transform` | $Z W + \mu$，线性可逆 |
| `PCA.explained_variance_ratio_` | 每个成分占总方差的比例 |
| `reconstruction_error` | $\mathrm{mean}((X-\hat X)^2)$，用来选 $k$ |
| `kernel_pca` | 中心化核矩阵上的 PCA；RBF 可拆同心圆 |

本课跳过 MNIST 下载以及 sklearn / t-SNE / UMAP 对照。要看那些完整打印 demo，需要额外依赖，运行：

```bash
python dim_reduction.py
```
